# PTV Analysis of the Tracked Sequences

This notebook consumes the cropped images produced by `opencv_tracker_v3.py` and:



In [ ]:
import csv
import json
import re
from importlib import import_module
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import trackpy as tp
if not hasattr(np, 'int'):
    np.int = int
import pandas as pd
import yaml
from IPython.display import display

plt.rcParams['figure.figsize'] = (6, 6)
CONFIG_PATH = Path('../configs/opencv_tracker_v3.yaml')

In [ ]:
def resolve_path(path_like, base_dir: Path) -> Path:
    candidate = Path(path_like)
    return candidate if candidate.is_absolute() else (base_dir / candidate).resolve()

def natural_sort_key(path: Path):
    parts = re.split(r'(\d+)', path.name)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def collect_images(folder: Path, allowed_exts=None):
    allowed_exts = [ext.lower() for ext in (allowed_exts or [])]
    candidates = [
        p
        for p in folder.iterdir()
        if p.is_file()
        and not p.name.startswith('.')
    ]
    if allowed_exts:
        candidates = [p for p in candidates if p.suffix.lower() in allowed_exts]
    return sorted(candidates, key=natural_sort_key)

def load_experiment_metadata(experiment_dir: Path) -> dict:
    for suffix in ('metadata.json', 'metadata.csv'):
        path = experiment_dir / suffix
        if path.exists():
            if path.suffix.lower() == '.json':
                return json.loads(path.read_text(encoding='utf-8'))
            with path.open('r', encoding='utf-8') as handle:
                reader = csv.DictReader(handle)
                for row in reader:
                    return row
    return {}




In [ ]:
# Load config files and paths

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config_dir = CONFIG_PATH.parent
input_cfg = config['input']
base_dir = resolve_path(input_cfg['base_dir'], config_dir)
raw_root_candidate = input_cfg.get('raw_root')
raw_root = resolve_path(raw_root_candidate or base_dir.parent, config_dir)
processed_root_candidate = input_cfg.get('processed_root')
fallback_processed = raw_root.parent / 'processed'
processed_root = resolve_path(processed_root_candidate or fallback_processed, config_dir)
try:
    relative = base_dir.relative_to(raw_root)
except ValueError:
    relative = Path(base_dir.name)
processed_experiment_dir = processed_root / relative
crops_subdir = config['output'].get('crops_subdir', 'cropped') or 'cropped'
crops_root = processed_experiment_dir / crops_subdir

piv_root = processed_experiment_dir / 'ptv_data'
piv_root.mkdir(parents=True, exist_ok=True)
ptv_summary_path = piv_root / 'ptv_summary.csv'
if ptv_summary_path.exists():
    ptv_summary_records = pd.read_csv(ptv_summary_path)
else:
    ptv_summary_records = pd.DataFrame()
ptv_summary_rows = []

summary_path = processed_experiment_dir / config['output'].get('summary_json', 'frame_rate_summary.json')
summary_data = json.loads(summary_path.read_text(encoding='utf-8'))
folder_summaries = summary_data.get('folder_summaries', [])
metadata_record = load_experiment_metadata(processed_experiment_dir)
tracking_csv = processed_experiment_dir / config['output'].get('metadata_csv', 'tracking_metadata.csv')
tracking_df = pd.read_csv(tracking_csv) if tracking_csv.exists() else pd.DataFrame()
print(f'Processed data directory: {processed_experiment_dir}')
print(f'Loaded {len(folder_summaries)} folder summaries and {len(tracking_df)} tracking rows.')


In [ ]:
def build_far_field_mask(image: np.ndarray, threshold: float, iterations: int = 5):
    _, mask = cv2.threshold(image, int(threshold), 255, cv2.THRESH_BINARY)
    if not np.any(mask):
        _, mask = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    expanded = cv2.dilate(mask, kernel, iterations=iterations)
    far_field = cv2.bitwise_not(expanded)
    return far_field, expanded

def histogram_stretch(image: np.ndarray, lower_pct: float = 0.1, upper_pct: float = 99.9) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    if arr.ndim != 2:
        arr = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
    low = np.percentile(arr, lower_pct)
    high = np.percentile(arr, upper_pct)
    if high <= low:
        return arr.astype(np.uint8)
    stretched = (arr - low) * (255.0 / (high - low))
    stretched = np.clip(stretched, 0, 255).astype(np.uint8)
    return stretched

def filter_tracks_by_mask(tracks, mask, ycol='y', xcol='x', particle_col='particle',
                          drop_entire_track=True):
    h, w = mask.shape

    yy = np.rint(tracks[ycol].to_numpy()).astype(int)
    xx = np.rint(tracks[xcol].to_numpy()).astype(int)

    yy = np.clip(yy, 0, h - 1)
    xx = np.clip(xx, 0, w - 1)

    inside = mask[yy, xx].astype(bool)

    out = tracks.copy()
    out['inside_mask'] = inside

    if drop_entire_track:
        bad_particles = out.groupby(particle_col)['inside_mask'].any()
        bad_particles = bad_particles[bad_particles].index
        out = out.loc[~out[particle_col].isin(bad_particles)].copy()
    else:
        out = out.loc[~out['inside_mask']].copy()

    return out.drop(columns='inside_mask')

def compute_particle_velocities(tracks):
    """
    Compute one mean velocity vector per particle from start/end positions,
    using elapsed frame intervals: (end_frame - start_frame).

    Parameters
    ----------
    tracks : pandas.DataFrame
        Must contain columns: particle, frame, x, y

    Returns
    -------
    particle_vel : pandas.DataFrame
        One row per particle with vx, vy, speed, and track summary info.
    """
    tracks = tracks.reset_index(drop=True).copy()

    rows = []

    for pid, grp in tracks.groupby('particle'):
        grp = grp.sort_values('frame')

        start = grp.iloc[0]
        end = grp.iloc[-1]

        start_frame = int(start['frame'])
        end_frame = int(end['frame'])
        n_steps = end_frame - start_frame

        # skip degenerate tracks
        if n_steps <= 0:
            continue

        dx = end['x'] - start['x']
        dy = end['y'] - start['y']

        vx = dx / n_steps
        vy = dy / n_steps

        rows.append({
            'particle': pid,
            'start_frame': start_frame,
            'end_frame': end_frame,
            'n_steps': n_steps,
            'x_start': start['x'],
            'y_start': start['y'],
            'x_end': end['x'],
            'y_end': end['y'],
            'dx': dx,
            'dy': dy,
            'vx': vx,
            'vy': vy,
            'speed': np.sqrt(vx**2 + vy**2),
        })

    return pd.DataFrame(rows)


def summarize_particle_velocities(particle_vel):
    """
    Average particle velocities across particles and return mean/std.
    """
    return {
        'n_particles': len(particle_vel),
        'mean_vx': particle_vel['vx'].mean(),
        'std_vx': particle_vel['vx'].std(ddof=1),
        'mean_vy': particle_vel['vy'].mean(),
        'std_vy': particle_vel['vy'].std(ddof=1),
        'mean_speed': particle_vel['speed'].mean(),
        'std_speed': particle_vel['speed'].std(ddof=1),
    }

In [ ]:
# PTV parameters

feature_size = 11
feature_minmass = 400

link_search_distance = 30
link_memory = 3
min_track_length = 10

max_frames = 50

In [ ]:
ptv_artifacts = {}
save_ext = config['output'].get('save_extension', '.png').lower()
allowed_exts = [save_ext]

for folder_summary in folder_summaries:
    folder_name = folder_summary.get('subfolder')
    threshold_value = folder_summary.get('threshold_value')
    frame_rate = folder_summary.get('frame_rate_hz')
    if not folder_name or threshold_value is None or not frame_rate:
        print(f"Skipping '{folder_name}' because of missing threshold/frame rate.")
        continue
    cropped_folder = crops_root / f"{folder_name}_cropped"
    if not cropped_folder.exists():
        print(f"Missing cropped folder: {cropped_folder}")
        continue
    image_paths = collect_images(cropped_folder, allowed_exts)
    if len(image_paths) < 2:
        print(f"Not enough frames in {cropped_folder} to compute PTV.")
        continue
    grayscale_images = [cv2.imread(str(path), cv2.IMREAD_GRAYSCALE) for path in image_paths]
    _, expanded_mask = build_far_field_mask(
        grayscale_images[0], threshold_value, iterations=20
    )
    expanded_mask = expanded_mask.astype('uint8', copy=False)
    dt = 1.0 / frame_rate
    min_dim = min(image.shape[0] for image in grayscale_images)

    pixel_per_um = float(metadata_record.get('pixelperum', 1.36)) or 1.36
    ptv_subdir = piv_root / f"{folder_name}_ptv_data"
    ptv_subdir.mkdir(parents=True, exist_ok=True)

    # Only process the first max_frames frames
    image_stretched_sequence = []
    for image in grayscale_images[:max_frames]:
        stretched = histogram_stretch(image, 0.1, 99.9)
        image_stretched_sequence.append(stretched)

    # Detect tracker particles and link into tracks
    features = tp.batch(image_stretched_sequence, feature_size, minmass=feature_minmass)
    tracks = tp.link(features, link_search_distance, memory=link_memory)
    tracks_filtered = tp.filter_stubs(tracks, min_track_length)

    # Drop tracks that enter the object mask at any point (i.e. keep only tracks that are fully outside the mask).
    tracks_filtered = filter_tracks_by_mask(tracks_filtered, expanded_mask, drop_entire_track=True)

    # Plot the tracks
    plt.figure()
    tp.plot_traj(tracks_filtered)

    particle_vel = compute_particle_velocities(tracks_filtered)
    summary = summarize_particle_velocities(particle_vel)
        
    mean_speed_pixels_per_frame = summary['mean_speed']
    std_speed_pixels_per_frame = summary['std_speed']

    mean_speed_um_per_s = mean_speed_pixels_per_frame * frame_rate * (1/pixel_per_um)
    std_speed_um_per_s = std_speed_pixels_per_frame * frame_rate * (1/pixel_per_um)

    print(f"mean speed: {mean_speed_um_per_s:.2f} um/s")
    print(f"std speed: {std_speed_um_per_s:.2f} um/s")

    # Persist tracks and particle velocities
    tracks_csv = ptv_subdir / 'tracks.csv'
    tracks_filtered.to_csv(tracks_csv, index=False)
    particle_vel_csv = ptv_subdir / 'particle_velocities.csv'
    particle_vel.to_csv(particle_vel_csv, index=False)

    summary_row = {
        'subfolder': folder_name,
        'threshold_value': threshold_value,
        'frame_rate_hz': frame_rate,
        'pixelperum': pixel_per_um,
        'objective': metadata_record.get('objective', ''),
        'particle_type': metadata_record.get('particle_type', ''),
        'mean_vx_um_per_s': summary['mean_vx'] * frame_rate * (1/pixel_per_um),
        'mean_vy_um_per_s': summary['mean_vy'] * frame_rate * (1/pixel_per_um),
        'std_vx_um_per_s': summary['std_vx'] * frame_rate * (1/pixel_per_um),
        'std_vy_um_per_s': summary['std_vy'] * frame_rate * (1/pixel_per_um),
        'mean_speed_um_per_s': mean_speed_um_per_s,
        'std_speed_um_per_s': std_speed_um_per_s,
        'n_particles': summary['n_particles']
    }
    ptv_summary_rows.append(summary_row)

# Write updated PTV summary csv
if ptv_summary_rows:
    updates = pd.DataFrame(ptv_summary_rows)
    if not ptv_summary_records.empty:
        to_drop = ptv_summary_records['subfolder'].isin(updates['subfolder'])
        ptv_summary_records = ptv_summary_records.loc[~to_drop]
    ptv_summary_records = pd.concat([ptv_summary_records, updates], ignore_index=True)
    ptv_summary_records.to_csv(ptv_summary_path, index=False)
    print(f"Wrote {len(updates)} rows to {ptv_summary_path}")


In [ ]:
frame_rate